In [1]:
import time
import os
import requests
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
# Set up Chrome WebDriver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# Open the BSSDB website
driver.get("https://www.bssdb.dev")

# Maximize the window
driver.maximize_window()

# Wait for the page to load
driver.implicitly_wait(5)

In [3]:
# Create a folder to store images
if not os.path.exists("card_thumbnails"):
    os.makedirs("card_thumbnails")

In [4]:
def extract_data_from_card():
    """
    Extract all relevant card data from the currently loaded web page.
    Returns a JSON object with basic card info, effects, and required cores.
    """
    card_data = {}

    # Wait for the "Card Type" element to be present
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "db-bss-zoom-details"))
        )
        
        # Extract basic card data
        card_data["name"] = driver.find_element(By.CSS_SELECTOR, ".db-bss-zoom-details .db-bss-zoom-head h2").text.strip()
        card_data["ID"] = driver.find_element(By.CSS_SELECTOR, ".game-zoom-image").get_attribute("src").split("/")[-1].split(".")[0]
        card_data["set"] = card_data["ID"].split("-")[-2]
        card_data["image"] = f"images/{card_data["set"]}/{card_data["ID"]}.png"
        card_data["color"] = driver.find_element(By.CSS_SELECTOR, ".db-bss-zoom-head .ui.red.mini.label").text.strip()
        card_data["cost"] = driver.find_element(By.XPATH, "//div[text()='Cost']/preceding-sibling::div[@class='value']").text

        # Attempt to find the element with the "Reduction" label
        reduction_element = driver.find_element(By.XPATH, "//div[text()='Reduction']/preceding-sibling::div[@class='value']")
        # Check if there is text (e.g., a dash for empty reduction)
        reduction_text = reduction_element.text.strip()
        if reduction_text:
            # If there's text, use it directly
            card_data["reduction"] = reduction_text
        else:
            # If there’s no text, look for images (symbols)
            symbols_elements = reduction_element.find_elements(By.XPATH, ".//img")
            card_data["reduction"] = [img.get_attribute('alt').split()[0].lower() for img in symbols_elements]

        # Extract symbols
        symbols = driver.find_elements(By.XPATH, "//div[@class='label' and text()='Symbols']/preceding-sibling::div[@class='value']//img")
        card_data["symbols"] = [img.get_attribute('alt').split()[0].lower() for img in symbols]

        card_data["cardType"] = driver.find_element(By.XPATH, "//div[text()='Card Type']/preceding-sibling::div[@class='value']").text
        card_data["subType"] = driver.find_element(By.XPATH, "//div[text()='Sub Type(s)']/preceding-sibling::div[@class='value']").text
        card_data["rarity"] = driver.find_element(By.XPATH, "//div[text()='Rarity']/preceding-sibling::div[@class='value']").text

    except Exception as e:
        print(f"Error extracting card data: {e}")

    # Extract required cores
    try:
        # Locate all elements for core requirements
        core_elements = driver.find_elements(By.XPATH, "//div[contains(@class, 'bss-core-bp')]")

        # Initialize the list to hold core requirements
        core_requirements = []

        # Loop through each core element and extract details
        for core_element in core_elements:
            # Extract level
            level_roman = core_element.find_element(By.XPATH, ".//span[@class='roman']").text.strip()
            # Convert Roman numeral to integer
            level = {"I": 1, "II": 2, "III": 3}.get(level_roman, 0)  # Add more as needed

            # Extract BP
            bp = int(core_element.find_element(By.XPATH, ".//div[@class='bp']").text.strip())

            # Extract cores
            cores = int(core_element.find_element(By.XPATH, ".//div[@class='cores']/span").text.strip())

            # Add to the list
            core_requirements.append({
                "level": level,
                "bp": bp,
                "cores": cores
            })

        # Assign the list to card_data
        card_data["coreRequirements"] = core_requirements

    except Exception as e:
        print(f"Error extracting required cores: {e}")

    # Extract effects
    try:
        effects = []
        effect_elements = driver.find_elements(By.CSS_SELECTOR, ".bss-card-effect")

        for effect in effect_elements:
            effect_data = {}

            # Extract levels
            level_imgs = effect.find_elements(By.CSS_SELECTOR, ".bss-card-effect-level")
            levels = [int(img.get_attribute("alt").replace("LV", "")) for img in level_imgs]
            effect_data["levels"] = levels if levels else []

            # Extract condition
            try:
                condition = effect.find_element(By.CSS_SELECTOR, ".bss-card-effect-condition").text.strip()
            except Exception:
                condition = "N/A"
            effect_data["condition"] = condition

            # Extract keywords
            keywords_data = []
            try:
                keyword_elements = effect.find_elements(By.CSS_SELECTOR, ".bss-card-effect-keyword")
                for keyword_element in keyword_elements:
                    keyword_text = keyword_element.text.strip()
                    if ":" in keyword_text:
                        keyword, rest = keyword_text.split(": ", 1)
                        target, value = rest.rsplit(" x", 1) if " x" in rest else (rest, None)
                    else:
                        keyword, target, value = keyword_text, None, None

                    keywords_data.append({
                        "keyword": keyword.strip(),
                        "target": target.strip() if target else None,
                        "value": int(value) if value else None
                    })
            except Exception:
                keywords_data = []
            effect_data["keywords"] = keywords_data

            # Extract details
            try:
                details = effect.find_element(By.CSS_SELECTOR, ".bss-card-effect-details").text.strip()
            except Exception:
                details = "N/A"
            effect_data["details"] = details

            # Add empty steps (as per your JSON example)
            effect_data["steps"] = []

            effects.append(effect_data)

        # Add extracted effects to card_data
        card_data["effects"] = effects

    except Exception as e:
        print(f"Error extracting effects: {e}")
        
    card_data["effects"] = effects


    # Save the extracted data as JSON
    try:
        folder_path = os.path.join("json", card_data["set"])
        os.makedirs(folder_path, exist_ok=True)
        file_path = os.path.join(folder_path, f"{card_data["ID"]}.json")
        with open(file_path, "w", encoding="utf-8") as json_file:
            json.dump(card_data, json_file, indent=4, ensure_ascii=False)
        print(f"Card data saved to {file_path}")
    except Exception as e:
        print(f"Error saving card data: {e}")

    return card_data

In [ ]:
# Function to download the images from the current page
def download_images():
    # Find the image elements based on the `img` tag with `src` containing the thumb URL
    images = driver.find_elements(By.CSS_SELECTOR, "img[src*='thumb']")  # This will match all images whose 'src' contains 'thumb'

    # Download each image and save to the corresponding folder
    for idx, img in enumerate(images):
        # Get the image URL
        img_url = img.get_attribute("src")
        
        # Extract the part of the URL that is used as the image name (e.g., BSS06-001)
        img_name = img_url.split("/")[-1].split(".")[0]  # Split at "/" to get the last part and remove the ".png"
        
        # Get the folder name from the first part of the image name (before the dash)
        folder_name = img_name.split("-")[0]
        
        # Create the folder if it does not exist
        folder_path = f"card_thumbnails/{folder_name}"
        os.makedirs(folder_path, exist_ok=True)
        
        # Download the image
        img_data = requests.get(img_url).content
        
        # Save the image to the folder using the extracted name
        img_path = os.path.join(folder_path, f"{img_name}.png")
        with open(img_path, "wb") as img_file:
            img_file.write(img_data)

In [6]:
# Start scraping images from the first page
#download_images()

In [7]:
# Function to cycle through each card on the page
def cycle_through_cards_on_page():
    try:
        # Wait for the card images to be visible on the page
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img"))
        )
        
        # Get all card images
        card_images = driver.find_elements(By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")
        
        for index, image in enumerate(card_images):
            try:
                # Re-locate the image to avoid stale element issues
                card_images = driver.find_elements(By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")
                image = card_images[index]

                # Scroll into view
                driver.execute_script("arguments[0].scrollIntoView({ behavior: 'smooth', block: 'center' });", image)

                # Wait for the image to be clickable and click it
                WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")))
                image.click()
                time.sleep(2)  # Wait for the card to open

                # Extract data from the card
                extract_data_from_card()

                # Wait for the close button and click it
                close_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "button.bp4-button.bp4-small.bp4-intent-danger"))
                )
                close_button.click()
                time.sleep(1)  # Allow time for the card to close

            except Exception as e:
                print(f"Error while interacting with card {index + 1}: {e}")

    except Exception as e:
        print(f"Error while cycling through cards: {e}")


In [8]:
def navigate_pages():
    # First, cycle through cards on the current page
    cycle_through_cards_on_page()

    while True:
        try:
            # Find the "next page" button by targeting the button class and icon
            next_button = driver.find_element(By.CSS_SELECTOR, "button.bp4-button.bp4-outlined.bp4-intent-primary span.bp4-icon-caret-right")
            
            # Scroll the "next" button into view to ensure it's clickable
            driver.execute_script("arguments[0].scrollIntoView();", next_button)

            # Check if the button is disabled (indicating the last page)
            if "bp4-disabled" in next_button.get_attribute("class"):
                print("Reached the last page.")
                break  # Exit the loop when the button is disabled
            
            # Wait until the next button is clickable and click it
            WebDriverWait(driver, 10).until(EC.element_to_be_clickable(next_button))
            next_button.click()
            
            # Wait for the page to load by checking for the presence of image elements
            WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")))

            # After the page loads, cycle through the cards again
            cycle_through_cards_on_page()

            #download_images()

            # Add a slight delay before the next page to avoid overwhelming the server
            time.sleep(2)
        
        except Exception as e:
            # If an error occurs (i.e., the next button is not found or any other issue), break the loop
            print("No more pages or error:", e)
            break


In [ ]:
navigate_pages()

Card data saved to json\BSS06\BSS06-001.json
Card data saved to json\BSS06\BSS06-002.json
Card data saved to json\BSS06\BSS06-002_p1.json
Card data saved to json\BSS06\BSS06-003.json


KeyboardInterrupt: 

In [ ]:
# Close the browser
driver.quit()